In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets who often enjoy their own personal space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals who need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [2]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets who often enjoy their own personal space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals who need plenty of space to hop around.')]

In [22]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(model= "llama-3.3-70b-versatile",groq_api_key=GROQ_API_KEY)

In [23]:
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002282FBBC450>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002282FC4C910>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1702.17it/s]


In [15]:
from langchain_chroma import Chroma

vectorStore = Chroma.from_documents(documents=documents, embedding=embeddings)

In [17]:
vectorStore.similarity_search("cat", k = 2)

[Document(id='9a9e78e6-ba93-494f-a6a6-d473c61d8811', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets who often enjoy their own personal space.'),
 Document(id='b0b26552-0015-48bb-8666-0feb75aac80a', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals who need plenty of space to hop around.')]

In [18]:
await vectorStore.asimilarity_search('rabbit', k = 2)

[Document(id='b0b26552-0015-48bb-8666-0feb75aac80a', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals who need plenty of space to hop around.'),
 Document(id='9a9e78e6-ba93-494f-a6a6-d473c61d8811', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets who often enjoy their own personal space.')]

In [13]:
vectorStore.similarity_search_with_score("dogs", k = 2)

[(Document(id='d189af75-9364-4b17-8e7e-fa210b4586f4', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.0435826778411865),
 (Document(id='a0b78c01-aba7-414b-bb5d-713a2226e473', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.0435826778411865)]

In [19]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorStore.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])

[[Document(id='9a9e78e6-ba93-494f-a6a6-d473c61d8811', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets who often enjoy their own personal space.')],
 [Document(id='18c00625-7add-46e2-930e-e544676a4b1f', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [20]:
retriever = vectorStore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k" : 1} 
)

retriever.batch(['parrot','rabbit'])


[[Document(id='4a185379-952e-48ab-aad3-d33ce34373ae', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')],
 [Document(id='b0b26552-0015-48bb-8666-0feb75aac80a', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals who need plenty of space to hop around.')]]

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Content:
{context}

"""

prompt = ChatPromptTemplate.from_messages(
    [("human",message)]
)

rag_chain = {"context":retriever, "question":RunnablePassthrough()}|prompt|llm

response = rag_chain.invoke("tell me about dogs")

response.content

'Dogs are great companions, known for their loyalty and friendliness.'